> 在线讲义：[第 12 章 · 转置与公式 1 完整走一遍](https://wikieden.github.io/transformer-visual/lessons/12-transpose-formula.html) · 本书《从零读懂 Transformer》

# 第 12 章 · 转置与公式 1 完整走一遍

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/12-transpose-formula.html) · 仓库里 `transformer-deep-dive/notes/lessons/12-transpose-formula.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 转置 | $K^\top$：行变列 | `K.T` | (3, 2) → (2, 3) |
| 形状对不上 | 左边列数 ≠ 右边行数 | `Q @ K` | 报错：shapes cannot be multiplied (3x2 and 3x2) |
| ① 打分 | $\mathbf q K^\top$ | `Q @ K.T` | 她：[0, 4, 2] |
| ② 缩放 | $\div\sqrt{2}$ | `/ math.sqrt(d_k)` | [0, 2.83, 1.41] |
| ③ 分比例 | softmax | `torch.softmax(..., dim=-1)` | [0.045, 0.768, 0.187] |
| ④ 取信息 | $\times V$ | `p @ V` | [8.61, 9.07] |
| 自带的公式 (1) | $\text{softmax}(QK^\top/\sqrt{d_k})V$ | `F.scaled_dot_product_attention(q, k, v)` | 和手写一样 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import torch
import torch.nn.functional as F
# 下面几行来自 ch11_matrix_qk.py（第 11 章）
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.0, 2.0]])  # 每一行是一个词的 q
K = torch.tensor([[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]])  # 每一行是一个词的 k
V = torch.tensor([[0.0, 10.0], [10.0, 10.0], [5.0, 5.0]])  # 小明、小红、她的 v

**函数 `attention`**：softmax(Q K^T / √d_k) V，每一步都留下来看。

In [ ]:
def attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> dict[str, torch.Tensor]:
    """softmax(Q K^T / √d_k) V，每一步都留下来看。"""
    d_k = q.shape[-1]
    scores = q @ k.T
    scaled = scores / math.sqrt(d_k)
    p = torch.softmax(scaled, dim=-1)
    return {"scores": scores, "scaled": scaled, "p": p, "out": p @ v}

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 3) -> list[float]:
    return [round(float(x), n) for x in t]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("K 形状", tuple(K.shape), "→ K.T 形状", tuple(K.T.shape))
print("K.T =\n", K.T, "\n（三列依次是小明、小红、她的 k）")
try:
    Q @ K
except RuntimeError as e:
    print("Q @ K（不转置）报错:", str(e).split("\n")[0])
steps = attention(Q, K, V)

In [ ]:
print("\n'她'那一行：")
print("  ① 打分   :", r(steps["scores"][2], 2))
print("  ② ÷ √2   :", r(steps["scaled"][2], 2))
print("  ③ softmax:", r(steps["p"][2]))
print("  ④ × V    :", r(steps["out"][2], 2))
print("整张输出（每行一个词的新向量）:\n", steps["out"])
builtin = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
print("PyTorch 自带的 F.scaled_dot_product_attention 结果一样:", torch.allclose(builtin, steps["out"], atol=1e-6))

## 3. 代码里才看得到的

1. **讲义 0.769、8.62，代码 0.768、8.61**：讲义手算时先把 2.828 四舍五入成 2.83、1.414 写成 1.41，再往下算，误差一路带到了最后；代码全程不四舍五入。两个都对，差在第三位小数。
2. **看懂报错**：`mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)`：左边 3 × 2 的列数 2 和右边 3 × 2 的行数 3 对不上。以后写 attention，十个错里有九个是形状。
3. **`Q[None]`**：在最前面加一维，变成 1 × 3 × 2（1 句话）。自带函数要求最前面是"第几句"，最后两维才是"词 × 数"。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 矩阵乘矩阵：左边取行，右边取列，各做一次点积；
> 转置就是行变列，K 翻过来每列一个 k，才能和 Q 对上；
> 左边列数等于右边行数，n × d 乘 d × n 得 n × n；
> 公式 1 四步：打分、缩放、softmax、乘 V。